[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yu-hsiu/QuaCCAToo/blob/main/verify_fix_colab.ipynb)


# QuaCCAToo 修復驗證(Colab)

驗證 `Yu-hsiu/QuaCCAToo` 的修復 commit `f201cf6`。

**缺陷**:`PulsedSim.run()` 原本一律用 QuTiP 的 `parallel_map`。工作程序只有在 fork 型平台才繼承父程序狀態;Windows / macOS 預設用 spawn,工作程序會重新 import 定義序列的模組,於是 import 之後才設定的 `rho0`、`observable`、`H2`、`c_ops` 在它們那邊全部不存在 —— 模擬跑的是另一個物理系統,而且不會報錯。

**修法**:`run()` 改為預設 `serial_map`,只有明確傳 `map_kw={"num_cpus": N}` 且 N > 1 才走 `parallel_map`,並在非 fork 平台發出警告。

**症狀**:`tests/test_qct.py::test_add_pulse` 的 pseudo-Hadamard 最佳 RF 相位,論文值 2.9 rad,缺陷下得到 2.7 rad。

## 1. 安裝

In [ ]:
!git clone https://github.com/Yu-hsiu/QuaCCAToo.git
%cd QuaCCAToo
!pip install -q -e . pytest

## 2. 跑測試套件

預期 `40 passed, 3 skipped`。

In [ ]:
!python -m pytest tests -q

## 3. 重點:上面那格證明不了修復有效

Colab 是 Linux,用 fork,這個 bug 在那裡本來就不會發作。修復前的程式在 Colab 也是 `40 passed`。

所以第 2 格只證明「修復沒有弄壞 Linux 的既有行為」。要真的重現並驗證,得強制切成 spawn 模式 —— 模擬 Windows / macOS 的行為。

In [ ]:
%%writefile /content/spawn_check.py
import multiprocessing, numpy as np
from qutip import basis, fock_dm, tensor
from quaccatoo import NV, PulsedSim

nv = NV(B0=25, units_B0="mT", N=14); nv.truncate(mS=1, mI=1)
w1_rf, w1_mwa, w1c = 0.2, 16, 2.14/3**0.5
tpi_rf, tpi_mwa, tpi_c = 1/(2*w1_rf), 1/(2*w1_mwa), 1/(2*w1c)
w0_rf, w0_mwa, w0_c = nv.RF_freqs[2], nv.MW_freqs[0], nv.energy_levels[2]
SOL = {"nsteps": 1e6}

def seq(phi, **kw):
    s = PulsedSim(nv)
    s.add_pulse(tpi_c,    w1c*nv.MW_h1,   pulse_params={"f_pulse": w0_c,   "phi_t": np.pi/2}, options=SOL)
    s.add_pulse(tpi_rf/2, w1_rf*nv.RF_h1, pulse_params={"f_pulse": w0_rf,  "phi_t": phi},     options=SOL)
    s.add_pulse(tpi_mwa,  w1_mwa*nv.MW_h1,pulse_params={"f_pulse": w0_mwa, "phi_t": np.pi/2}, options=SOL)
    s.add_pulse(tpi_rf/2, w1_rf*nv.RF_h1, pulse_params={"f_pulse": w0_rf,  "phi_t": phi},     options=SOL)
    return s.rho

if __name__ == "__main__":
    multiprocessing.set_start_method("spawn", force=True)   # 模擬 Windows / macOS
    nv.rho0 = tensor(basis(2,0)-basis(2,1), basis(2,0)-basis(2,1)).unit()
    nv.observable = [tensor(fock_dm(2,0),fock_dm(2,0)), tensor(fock_dm(2,0),fock_dm(2,1)),
                     tensor(fock_dm(2,1),fock_dm(2,0)), tensor(fock_dm(2,1),fock_dm(2,1))]
    phis = np.arange(1.5, 3.3, 0.1)
    for label, mk in [("serial (修復後預設)", None), ("parallel num_cpus=2 (修復前行為)", {"num_cpus": 2})]:
        sim = PulsedSim(nv); sim.run(phis, seq, map_kw=mk)
        m = sim.results[0]**2 + sim.results[3]**2
        print(f"{label}: argmax phi = {phis[np.argmax(m)]:.2f} rad")

In [ ]:
!python /content/spawn_check.py

## 4. 判讀

| 輸出行 | 預期 | 意義 |
|---|---|---|
| `serial (修復後預設)` | **2.90 rad** | 論文值,修復後的預設路徑正確 |
| `parallel num_cpus=2 (修復前行為)` | **2.70 rad** + 警告訊息 | 重現缺陷 |

兩行數字不同,就是缺陷與修復的直接證據。

**誠實的但書**:`multiprocessing.set_start_method("spawn")` 是否真能改變 QuTiP 內部 executor 的行為,尚未實測 —— QuTiP 的 `parallel_map` 用 loky,它可能自己管理 context 而忽略全域設定。

如果 parallel 那行也印出 2.90,代表 loky 沒吃這個設定(不是修復失效),改用下一格的環境變數再試。

In [ ]:
# 備援:用環境變數強制 loky 使用 spawn
!LOKY_START_METHOD=spawn python /content/spawn_check.py